# 0. Full Pipeline

This notebook combines local setup, training entry points, and analysis for the next-scale AttnRes experiments without Google Drive.


In [6]:
!git clone https://github.com/AtinChing/AttnResGPT-next-scale /content/AttnResGPT-next-scale


fatal: destination path '/content/AttnResGPT-next-scale' already exists and is not an empty directory.


In [2]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if (candidate / 'src' / 'training' / 'train.py').exists() and (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find AttnResGPT-next-scale. Open the notebook from the repo root or sync the repo to /content/AttnResGPT-next-scale.'
    )

REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


Using repo at: /content/AttnResGPT-next-scale


In [3]:
import torch

print('cuda_available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device_name =', torch.cuda.get_device_name(0))
    print('bf16_supported =', torch.cuda.is_bf16_supported())


cuda_available = True
device_name = Tesla T4
bf16_supported = True


## Stage 1: Required First-Run Experiment

This runs SMALL only, contexts 128 and 512, baseline plus AttnRes, 300 steps each.


In [4]:
!python experiments/scale_experiment.py --config configs/first_run.yaml --skip-existing


Traceback (most recent call last):
  File "/content/AttnResGPT-next-scale/experiments/scale_experiment.py", line 165, in <module>
    main()
  File "/content/AttnResGPT-next-scale/experiments/scale_experiment.py", line 157, in main
    summary_rows.append(_run_one(run_config, skip_existing=args.skip_existing))
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AttnResGPT-next-scale/experiments/scale_experiment.py", line 112, in _run_one
    return train_from_config(config)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/AttnResGPT-next-scale/src/training/train.py", line 187, in train_from_config
    raise FileExistsError(
FileExistsError: Run directory already contains outputs for tinystories_small_baseline_ctx512_steps300_seed1337. Use training.resume_from to continue or remove the old outputs first.


In [5]:
import pandas as pd
summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
consolidated_df = pd.read_csv('outputs/logs/consolidated_summary_table.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')
display(consolidated_df.sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))


FileNotFoundError: [Errno 2] No such file or directory: 'outputs/logs/run_summaries.csv'

## Optional Sweeps

Uncomment the sweep you want to run after the first-run path is stable.


In [ ]:
!python experiments/scale_experiment.py --config configs/small.yaml --skip-existing
!python experiments/scale_experiment.py --config configs/medium.yaml --skip-existing
!python experiments/scale_experiment.py --config configs/large.yaml --skip-existing


## Analysis

This section reproduces the analysis notebook inside the same local workflow.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def find_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AttnResGPT-next-scale')]
    for candidate in candidates:
        if (candidate / 'src' / 'training' / 'train.py').exists() and (candidate / 'requirements.txt').exists():
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find AttnResGPT-next-scale. Open the notebook from the repo root or sync the repo to /content/AttnResGPT-next-scale.'
    )

REPO_DIR = find_repo_root()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(f'Using repo at: {REPO_DIR}')


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
summary_df = pd.read_csv('outputs/logs/run_summaries.csv')
consolidated_df = pd.read_csv('outputs/logs/consolidated_summary_table.csv')
paired_df = pd.read_csv('outputs/logs/paired_comparisons.csv')
display(consolidated_df.sort_values(['size', 'context', 'model']))
display(paired_df.sort_values(['size', 'context']))
if Path('outputs/summary_large.csv').exists():
    display(pd.read_csv('outputs/summary_large.csv').sort_values(['context', 'model']))
if Path('outputs/summary_large_comparison.csv').exists():
    display(pd.read_csv('outputs/summary_large_comparison.csv').sort_values(['context']))


In [ ]:
plot_dir = Path('outputs/plots')
plot_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=summary_df, x='context', y='val_loss', hue='model', style='size', marker='o', ax=axes[0])
axes[0].set_title('Validation Loss vs Context')
sns.lineplot(data=summary_df, x='context', y='perplexity', hue='model', style='size', marker='o', ax=axes[1])
axes[1].set_title('Perplexity vs Context')
fig.tight_layout()
fig.savefig(plot_dir / 'loss_and_perplexity_vs_context.png', dpi=200)
plt.show()


In [ ]:
attnres_rows = summary_df[summary_df['model'] == 'attnres'][['run_name', 'context']].sort_values('context')
for _, row in attnres_rows.iterrows():
    summary_path = Path('outputs/runs') / row['run_name'] / 'run_summary.json'
    payload = json.loads(summary_path.read_text(encoding='utf-8'))
    heatmap = payload.get('depth_attention_rows', [])
    if not heatmap:
        continue
    plt.figure(figsize=(8, 4))
    sns.heatmap(heatmap, cmap='viridis')
    plt.title(f"AttnRes depth attention: {row['run_name']}")
    plt.xlabel('Source index')
    plt.ylabel('Depth-mixing row')
    plt.tight_layout()
    output_path = plot_dir / f"depth_heatmap_{row['run_name']}.png"
    plt.savefig(output_path, dpi=200)
    plt.show()
